# Explore the staged Green Taxi file (phase 2)
Exploring only. The pipeline never imports or runs this notebook. Building and loading the tables happens in dbt.

In [1]:
import os
from pathlib import Path

import duckdb

# The notebook runs from notebooks/ and .env is not loaded here, so find the project root
# (the folder with pyproject.toml) and build every path from it.
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)


def env_value(name, default=None):
    """Environment first, then the .env file in the project root."""
    if name in os.environ:
        return os.environ[name]
    env_file = root / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith(f"{name}="):
                return line.split("=", 1)[1].strip()
    return default


staging = root / env_value("STAGING_DIR", "data/raw")
file = staging / "green_tripdata_2025-01.parquet"
src = f"read_parquet('{file}')"
con = duckdb.connect()
print("project root:", root)
print("exploring:", file.name, file.exists())

project root: /Users/jolenndarensbourg/code/jolennd/bootcamp/neueFische/projects/mle-data-pipeline
exploring: green_tripdata_2025-01.parquet True


## Columns and types

In [2]:
con.sql(f"describe select * from {src}").df()[["column_name", "column_type"]]

,column_name,column_type
0,VendorID,INTEGER
1,lpep_pickup_datetime,TIMESTAMP
2,lpep_dropoff_datetime,TIMESTAMP
3,store_and_fwd_flag,VARCHAR
4,RatecodeID,BIGINT
5,PULocationID,INTEGER
6,DOLocationID,INTEGER
7,passenger_count,BIGINT
8,trip_distance,DOUBLE
9,fare_amount,DOUBLE


## Row count (expected value for A1)

In [3]:
con.sql(f"select count(*) as rows from {src}").df()

,rows
0,48326


## Missing values in the columns the output needs, and in the key columns

In [4]:
con.sql(f"""select count(*) filter (where lpep_pickup_datetime is null) as null_pickup,
  count(*) filter (where total_amount is null) as null_total_amount,
  count(*) filter (where ehail_fee is null) as null_ehail_fee,
  count(*) filter (where payment_type is null) as null_payment_type,
  count(*) filter (where RatecodeID is null) as null_ratecode,
  count(*) filter (where VendorID is null) as null_vendor
from {src}""").df()

,null_pickup,null_total_amount,null_ehail_fee,null_payment_type,null_ratecode,null_vendor
0,0,0,48326,1836,1836,0


## Value ranges

In [5]:
con.sql(f"""select min(lpep_pickup_datetime) as first_pickup, max(lpep_pickup_datetime) as last_pickup,
  min(total_amount) as min_total, max(total_amount) as max_total, round(sum(total_amount), 2) as sum_total,
  count(*) filter (where total_amount < 0) as negative, count(*) filter (where total_amount = 0) as zero
from {src}""").df()

,first_pickup,last_pickup,min_total,max_total,sum_total,negative,zero
0,2024-12-25 23:13:15,2025-02-05 18:46:24,-114.0,371.4,1093822.36,153,33


## Pickup dates outside the file's own month (ADR 2)
Before 2025-01-01 is outside the range and excluded in staging. From 2025-02-01 is outside the file's month but inside the range and kept.

In [6]:
con.sql(f"""select cast(lpep_pickup_datetime as date) as pickup_date, count(*) as rows
from {src} where lpep_pickup_datetime < '2025-01-01' or lpep_pickup_datetime >= '2025-02-01'
group by 1 order by 1""").df()

,pickup_date,rows
0,2024-12-25,1
1,2024-12-29,1
2,2024-12-31,4
3,2025-02-01,19
4,2025-02-03,6
5,2025-02-04,7
6,2025-02-05,5


## Natural key: is any row repeated on all columns?

In [7]:
con.sql(
    f"""select count(*) as rows, (select count(*) from (select distinct * from {src})) as distinct_rows from {src}"""
).df()

,rows,distinct_rows
0,48326,48326


## Negative amounts: does each have a positive twin?
Twin: same pickup time, same pickup location, opposite amount.

In [8]:
con.sql(f"""select count(*) as negatives,
  count(*) filter (where exists (select 1 from {src} p where p.total_amount = -n.total_amount
    and p.lpep_pickup_datetime = n.lpep_pickup_datetime and p.PULocationID = n.PULocationID)) as with_twin
from {src} n where total_amount < 0""").df()

,negatives,with_twin
0,153,153


## Payment types and their revenue share

In [9]:
con.sql(f"""select payment_type, count(*) as trips, round(sum(total_amount), 2) as revenue,
  round(100 * sum(total_amount) / (select sum(total_amount) from {src}), 2) as pct_of_revenue
from {src} group by 1 order by 1""").df()

,payment_type,trips,revenue,pct_of_revenue
0,1,34635,808293.98,73.90
1,2,11424,227182.19,20.77
2,3,332,1522.45,0.14
3,4,98,202.00,0.02
4,5,1,32.45,0.00
5,<NA>,1836,56589.29,5.17


## Do the fee columns add up to total_amount?

In [10]:
con.sql(f"""select count(*) filter (where abs(coalesce(fare_amount,0) + coalesce(extra,0) + coalesce(mta_tax,0)
  + coalesce(tip_amount,0) + coalesce(tolls_amount,0) + coalesce(ehail_fee,0) + coalesce(improvement_surcharge,0)
  + coalesce(congestion_surcharge,0) + coalesce(cbd_congestion_fee,0) - total_amount) > 0.005) as rows_that_differ
from {src}""").df()

,rows_that_differ
0,6698


## Dimensions and measures
| Column | Role | Why |
|---|---|---|
| `pickup_date` (from `lpep_pickup_datetime`) | dimension, the grain of the mart | sets the day of a trip (ADR 2) |
| `payment_type` | dimension | how the passenger paid |
| `VendorID` | dimension | who reported the trip |
| `RatecodeID` | dimension | which fare rule applied |
| `PULocationID`, `DOLocationID` | dimension keys, no table yet | need the zone lookup file (later option) |
| `total_amount` | measure, additive | revenue, summed as is (ADR 1) |
| `trip_distance` | measure, additive | miles |
| `passenger_count` | measure, additive | people |

Not used: the fee columns (do not always add up to `total_amount`), `ehail_fee` (all null), flags and `trip_type`.

## Fact and dimension design (ADR 13)
```
raw_green_trips -> stg_green_trips -> fct_trips ----> mart_daily_revenue -> daily_revenue.parquet
                                         |  keys
              dim_date (generated) ------+
              dim_payment_type (seed) ---+
              dim_vendor (seed) ---------+
              dim_ratecode (seed) -------+
```
- `fct_trips`: one row per trip. `trip_id` is a hash of the partition and all source columns, and must be unique.
- A null key in the source becomes `-1` (Unknown) so every fact row has a dimension row.
- `mart_daily_revenue`: one row per `pickup_date` with `revenue` and `trip_count`, built from `fct_trips`.

## Preview: the rows that will feed fct_trips
Only a preview of the idea. The real build and load happen in dbt.

In [11]:
con.sql(f"""select cast(lpep_pickup_datetime as date) as pickup_date,
  coalesce(payment_type, -1) as payment_type, coalesce(VendorID, -1) as vendor_id,
  coalesce(RatecodeID, -1) as ratecode_id, PULocationID as pickup_location_id, DOLocationID as dropoff_location_id,
  cast(total_amount as decimal(12, 2)) as total_amount, trip_distance, passenger_count
from {src}
where lpep_pickup_datetime >= '2025-01-01' and lpep_pickup_datetime < '2025-04-01'
order by lpep_pickup_datetime limit 10""").df()

,pickup_date,payment_type,vendor_id,ratecode_id,pickup_location_id,dropoff_location_id,total_amount,trip_distance,passenger_count
0,2025-01-01,2,2,1,95,130,20.90,3.90,1
1,2025-01-01,1,2,1,74,262,27.54,2.91,1
2,2025-01-01,1,2,1,75,235,34.00,5.93,1
3,2025-01-01,2,2,1,243,116,19.50,3.22,1
4,2025-01-01,1,2,1,41,74,12.48,1.05,1
5,2025-01-01,2,2,5,171,73,26.55,0.41,1
6,2025-01-01,1,2,5,135,135,20.00,0.00,1
7,2025-01-01,1,2,1,95,135,20.00,1.98,1
8,2025-01-01,1,2,1,260,83,12.80,1.08,1
9,2025-01-01,1,2,1,223,223,10.79,0.75,5


In [12]:
con.sql(f"""select count(*) as rows_in_range, count(*) filter (where payment_type is null) as unknown_payment,
  round(sum(total_amount), 2) as total_in_range
from {src} where lpep_pickup_datetime >= '2025-01-01' and lpep_pickup_datetime < '2025-04-01' """).df()

,rows_in_range,unknown_payment,total_in_range
0,48320,1836,1093642.19


## After the build: inspect staging and the mart
These cells read the DuckDB file named by `DB_CONN`. They print a note and stop if the file or the tables do not exist yet. The fact table joins in after Phase 3 (ADR 13).

In [13]:
from data_pipeline.db import duckdb_path

db_path = Path(duckdb_path(env_value("DB_CONN", "duckdb:///data/warehouse.duckdb")))
db_path = db_path if db_path.is_absolute() else root / db_path
built = None
if db_path.exists():
    built = duckdb.connect(str(db_path), read_only=True)
    found = built.execute(
        "select table_schema, table_name from information_schema.tables "
        "where table_name in ('stg_green_trips', 'mart_daily_revenue') order by 2"
    ).df()
    if len(found) < 2:
        print("tables not built yet; found:", list(found["table_name"]))
        built = None
    else:
        print(found)
else:
    print("database file not found yet:", db_path)

  table_schema          table_name
0         main  mart_daily_revenue
1         main     stg_green_trips


In [14]:
if built is not None:
    display(
        built.execute(
            "select 'stg_green_trips' as t, count(*) as rows, sum(total_amount) as total from stg_green_trips "
            "union all select 'mart_daily_revenue', count(*), sum(revenue) from mart_daily_revenue"
        ).df()
    )

,t,rows,total
0,stg_green_trips,48320,1093642.19
1,mart_daily_revenue,35,1093642.19


In [15]:
if built is not None:
    display(
        built.execute(
            "select * from mart_daily_revenue order by pickup_date limit 10"
        ).df()
    )
    display(built.execute("select * from stg_green_trips limit 5").df())

,pickup_date,revenue,trip_count
0,2025-01-01,25076.51,1001
1,2025-01-02,34256.83,1516
2,2025-01-03,36517.69,1525
3,2025-01-04,27869.20,1249
4,2025-01-05,25823.69,1129
5,2025-01-06,34049.13,1541
6,2025-01-07,36844.24,1648
7,2025-01-08,37905.12,1680
8,2025-01-09,42532.87,1868
9,2025-01-10,37417.57,1633


,vendor_id,pickup_datetime,dropoff_datetime,pickup_date,store_and_fwd_flag,ratecode_id,pickup_location_id,dropoff_location_id,passenger_count,trip_distance,...,tolls_amount,improvement_surcharge,total_amount,is_total_amount_missing,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,loaded_at,source_partition
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,2025-01-01,N,1,75,235,1,5.93,...,0.00,1.0,34.00,False,1,1,0.00,0.0,2026-11-05 13:32:10.006956,2025-01
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,2025-01-01,N,1,166,75,1,1.32,...,0.00,1.0,11.10,False,2,1,0.00,0.0,2026-11-05 13:32:10.006956,2025-01
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,2025-01-01,N,5,171,73,1,0.41,...,0.00,1.0,26.55,False,2,2,0.00,0.0,2026-11-05 13:32:10.006956,2025-01
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,2025-01-01,N,1,74,223,1,4.12,...,6.94,1.0,36.77,False,1,1,0.00,0.0,2026-11-05 13:32:10.006956,2025-01
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,2025-01-01,N,1,66,158,1,4.71,...,0.00,1.0,46.86,False,1,1,2.75,0.0,2026-11-05 13:32:10.006956,2025-01
